In [ ]:
# Import necessary libraries
import rospy
import actionlib
from assignment_2_2024.msg import PlanningAction, PlanningGoal
from assignment_2_2024.msg import PositionVelocity
from assignment_2_2024.srv import GetLastTarget
from nav_msgs.msg import Odometry
from sensor_msgs.msg import LaserScan
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import math 
import matplotlib.pyplot as plt
from collections import deque
import time 

Ros Initilization

In [ ]:
if not rospy.core.is_initialized():
	rospy.init_node('action_client_gui_node', anonymous=True)

client = actionlib.SimpleActionClient('/reaching_goal', PlanningAction)
rospy.loginfo("Waiting for action server...")
client.wait_for_server()
rospy.loginfo("Connected to action server!")
goal = PlanningGoal()


GUI Widgets

In [ ]:
x_input = widgets.FloatText(description='X:', value=0.0)
y_input = widgets.FloatText(description='Y:', value=0.0)
send_button = widgets.Button(description='Send Goal', button_style='success')
cancel_button = widgets.Button(description='Cancel Goal', button_style='warning')
pos_output = widgets.Label()
vel_output = widgets.Label()
obs_output = widgets.Label()

display(widgets.HBox([x_input, y_input, send_button, cancel_button]))
display(pos_output, vel_output, obs_output)


Plotting Setup

In [ ]:
max_len = 100
x_traj = deque(maxlen=max_len)
y_traj = deque(maxlen=max_len)
velocities = deque(maxlen=max_len)
obstacles = deque(maxlen=max_len)
time_stamps = deque(maxlen=max_len)
obstacle_timestamps = deque(maxlen=max_len)

plt.ioff()
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 12))
traj_plot = ax1.plot([], [], 'b.-')[0]
vel_plot = ax2.plot([], [], 'r-')[0]
obs_plot = ax3.plot([], [], 'g-')[0]

ax1.set_title("Trajectory (x vs y)")
ax1.set_xlabel("x [m]")
ax1.set_ylabel("y [m]")
ax1.grid(True)

ax2.set_title("Linear Velocity over Time")
ax2.set_ylabel("v [m/s]")
ax2.grid(True)

ax3.set_title("Closest Obstacle Distance over Time")
ax3.set_ylabel("Distance [m]")
ax3.set_xlabel("Time [s]")
ax3.grid(True)

plt.tight_layout()
plot_output = widgets.Output()
display(plot_output)

Callbacks

In [ ]:
def feedback_cb(feedback):
    rospy.loginfo(f"Feedback: {feedback}")

def odom_callback(msg):
    t = time.time()
    x = msg.pose.pose.position.x
    y = msg.pose.pose.position.y
    vx = msg.twist.twist.linear.x

    x_traj.append(x)
    y_traj.append(y)
    velocities.append(vx)
    time_stamps.append(t)

    pos_output.value = f"Position: x={x:.2f}, y={y:.2f}"
    vel_output.value = f"Velocity: vx={vx:.2f}"

def scan_callback(msg):
    t = time.time()
    valid_ranges = [r for r in msg.ranges if not math.isinf(r)]
    if valid_ranges:
        min_dist = min(valid_ranges)
        obstacles.append(min_dist)
        obstacle_timestamps.append(t)
        obs_output.value = f"Closest Obstacle: {min_dist:.2f} m"

def update_plots(_):
    if len(x_traj) < 2:
        return
    with plot_output:
        clear_output(wait=True)

        traj_plot.set_data(x_traj, y_traj)
        ax1.relim()
        ax1.autoscale_view()

        vel_plot.set_data(time_stamps, velocities)
        ax2.relim()
        ax2.autoscale_view()

        obs_plot.set_data(obstacle_timestamps, obstacles)
        ax3.relim()
        ax3.autoscale_view()

        fig.canvas.draw()
        display(fig)


GUI Logic

In [ ]:
def send_goal_callback(b):
    x = x_input.value
    y = y_input.value
    goal.target_pose.header.frame_id = "map"
    goal.target_pose.header.stamp = rospy.Time.now()
    goal.target_pose.pose.position.x = x
    goal.target_pose.pose.position.y = y
    goal.target_pose.pose.orientation.w = 1.0
    client.send_goal(goal, feedback_cb=feedback_cb)
    rospy.loginfo(f"Goal sent: ({x}, {y})")

def cancel_goal_callback(b):
    client.cancel_goal()
    rospy.loginfo("Goal canceled")

Ros Subscribers

In [ ]:
rospy.Subscriber('/odom', Odometry, odom_callback)
rospy.Subscriber('/scan', LaserScan, scan_callback)

Bind Buttons

In [ ]:
send_button.on_click(send_goal_callback)
cancel_button.on_click(cancel_goal_callback)

Ros Thread

In [ ]:
def ros_spin():
    rate = rospy.Rate(2)  # update rate: 2 Hz
    while not rospy.is_shutdown():
        update_plots(None)
        rate.sleep()

ros_thread = threading.Thread(target=ros_spin)
ros_thread.start()